# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Maryam-Yaqoob/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

**Finding A — "What Predicts Health?" (Random Forest feature importance, ML Appendix, p.26).**
The paper reports Average Position as the top predictor of Health Score (43% importance),
followed by Impressions (32%) and Scroll Depth (15%), using an 80/20 holdout split. But the
paper's own methodology section states the Health Score target is *defined* as
Impressions(30pts) + Position(30pts) + CTR(20pts) + Scroll Depth(20pts) — meaning three of the
model's top features are literal components of the formula it's predicting. **My question:** does
an 80/20 holdout split protect against overfitting to noise, sure — but can it protect against a
target that's *definitionally* built from its own top features? No split design fixes that; the
paper is right to call the result "descriptive rather than causal," but I'd go further — this
isn't just non-causal, it's closer to a tautology check (confirming the formula's own weights)
than a genuine predictive-power test.

**Finding B — "What Predicts Growth?" (Logistic Regression, 71% holdout accuracy, ML Appendix,
p.28).** The paper reports Content Age as the strongest negative growth signal and Days Visible
as a strong positive one, with an 80/20 split. **My question:** was that 80/20 split done on
*rows* (content pieces) or *grouped by brand/client*? If it's a random row split across 57 brands,
a brand with a distinctive content strategy (consistently younger, consistently fresher pages)
could have pieces in both train and test, letting the model partly learn "which brand is this"
rather than a generalizable age/freshness pattern — the same client-leakage risk my own
`w05_model.ipynb` deliberately closes with a client-grouped split.


In [1]:
# No warehouse query needed for this section -- it's a methodology read of the paper, not new
# computation. Documenting the two claims and my questions as data for the record.
paper_findings = [
    {
        "finding": "Random Forest feature importance -> Health Score",
        "reported_result": "Avg Position 43%, Impressions 32%, Scroll Depth 15% importance (80/20 split)",
        "my_question": "Target is partly a linear formula OF these features -- does any split fix that?",
    },
    {
        "finding": "Logistic Regression -> growth vs decline",
        "reported_result": "71% holdout accuracy; Content Age strongest negative signal (80/20 split)",
        "my_question": "Row-level or brand-grouped split? 57 brands -> risk of brand leakage across split.",
    },
]
import pandas as pd
pd.DataFrame(paper_findings)


,finding,reported_result,my_question
0,Random Forest feature importance -> Health Score,"Avg Position 43%, Impressions 32%, Scroll Dept...",Target is partly a linear formula OF these fea...
1,Logistic Regression -> growth vs decline,71% holdout accuracy; Content Age strongest ne...,Row-level or brand-grouped split? 57 brands ->...


## 2. My model under an honest split (before/after)

`w05_model.ipynb` already used a **client-grouped** split throughout (70% of clients → train,
30% → test, never mixing one client's pages across both sides) — so there isn't a "sloppy"
version of my own model to contrast against. Instead, the honest before/after here is the
**baseline rule vs the model**, both scored on the *same* held-out clients:

- **Rule baseline** (`w04_baseline_score.ipynb`, whole month, not client-held-out):
  Precision@10 = 50.0% vs a 32.7% base rate.
- **Model** (`w05_model.ipynb`, refit on train clients only, scored on held-out test clients):
  Precision@10, Precision@50, and ROC-AUC — pull the actual printed numbers from your `w05_model`
  run below, since that's the fair, apples-to-apples comparison this section needs.


In [2]:
# ---- Setup: same DuckDB + HF pattern as w01/w03-w05. Run in Colab with your HF_TOKEN. ----
%pip -q install duckdb scikit-learn
import duckdb, pandas as pd, numpy as np

from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"
FACT = f"read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet')"
COLS = {"impressions": "gsc_impressions", "clicks": "gsc_clicks", "position": "gsc_avg_position"}

# ---- FIX: LEFT JOIN + COALESCE(...,0), matching w05_model.ipynb. The old INNER JOIN
# silently dropped any (client, content) pair with zero second-half activity at all. ----
feature_frame = con.sql(f"""
    WITH first_half AS (
        SELECT client_hash_id, content_hash_id,
               SUM({COLS['impressions']}) AS impressions_first_half,
               SUM({COLS['clicks']})      AS clicks_first_half,
               AVG(CASE WHEN {COLS['impressions']} > 0 THEN {COLS['position']} END) AS avg_position_first_half,
               COUNT(DISTINCT CASE WHEN {COLS['impressions']} > 0 THEN report_date END) AS active_days_first_half
        FROM {FACT}
        WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
        GROUP BY 1, 2
    ),
    second_half AS (
        SELECT client_hash_id, content_hash_id,
               SUM({COLS['impressions']}) AS impressions_second_half
        FROM {FACT}
        WHERE report_date BETWEEN DATE '2026-03-16' AND DATE '2026-03-31'
        GROUP BY 1, 2
    )
    SELECT
        f.client_hash_id, f.content_hash_id,
        f.impressions_first_half, f.clicks_first_half,
        f.clicks_first_half * 100.0 / NULLIF(f.impressions_first_half, 0) AS ctr_first_half,
        f.avg_position_first_half, f.active_days_first_half,
        COALESCE(s.impressions_second_half, 0) AS impressions_second_half,
        CASE WHEN COALESCE(s.impressions_second_half, 0) < 0.8 * f.impressions_first_half
             THEN 1 ELSE 0 END AS is_declining_next_half
    FROM first_half f
    LEFT JOIN second_half s USING (client_hash_id, content_hash_id)
    WHERE f.impressions_first_half > 0
""").df()

# ---- FIX: lock in a fixed row order so a tied "top 10" can't silently change between runs. ----
feature_frame = feature_frame.sort_values(["client_hash_id", "content_hash_id"]).reset_index(drop=True)
print(f"Feature frame: {len(feature_frame):,} rows")

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np

HONEST_FEATURES = ["impressions_first_half", "clicks_first_half", "ctr_first_half",
                    "avg_position_first_half", "active_days_first_half"]

# ---- FIX: kind="stable" so tied scores always break the same way, run to run. ----
def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores), kind="stable")
    return np.asarray(y_true)[order][:k].mean()

df = feature_frame.dropna(subset=HONEST_FEATURES).copy()
y = df["is_declining_next_half"]

train_clients, test_clients = train_test_split(
    df["client_hash_id"].unique(), test_size=0.3, random_state=42)
train_mask = df["client_hash_id"].isin(train_clients)
test_mask = df["client_hash_id"].isin(test_clients)

X_train, X_test = df.loc[train_mask, HONEST_FEATURES], df.loc[test_mask, HONEST_FEATURES]
y_train, y_test = y[train_mask], y[test_mask]

# ---- FIX: scale features first -- see capstone.ipynb / w05_model.ipynb for why. ----
scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr = LogisticRegression(max_iter=1000, random_state=42).fit(X_train_scaled, y_train)
lr_scores = lr.predict_proba(X_test_scaled)[:, 1]

print("Model (Logistic Regression), client-held-out test set:")
print(f"  Precision@10: {precision_at_k(y_test, lr_scores, 10):.1%}")
print(f"  Precision@50: {precision_at_k(y_test, lr_scores, 50):.1%}")
print()
print("Baseline (from w04_baseline_score.ipynb, whole month, NOT client-held-out):")
print("  Precision@10: 50.0%")
print("\n(Note the baseline number above isn't on the same held-out clients -- that gap")
print(" itself is worth naming as a validation-design honesty point, not just a scoreboard.)")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame: 151,981 rows
Model (Logistic Regression), client-held-out test set:
  Precision@10: 40.0%
  Precision@50: 28.0%

Baseline (from w04_baseline_score.ipynb, whole month, NOT client-held-out):
  Precision@10: 50.0%

(Note the baseline number above isn't on the same held-out clients -- that gap
 itself is worth naming as a validation-design honesty point, not just a scoreboard.)


## 3. Leakage audit

Same hunt as `w03_feature_leakage_check.ipynb`, re-run on the final feature set actually used in
`w05_model.ipynb`: `impressions_first_half`, `clicks_first_half`, `ctr_first_half`,
`avg_position_first_half`, `active_days_first_half`. All five are Mar 1-15 only.
`impressions_second_half` and `is_declining_next_half` were confirmed disjoint from the score
inputs in both `w04_baseline_score.ipynb`'s explicit assert and `w03_feature_leakage_check.ipynb`'s
Attack #1. Re-asserting here as the final gate before the paper cites these numbers.


In [3]:
FINAL_FEATURES = ["impressions_first_half", "clicks_first_half", "ctr_first_half",
                   "avg_position_first_half", "active_days_first_half"]
LABEL_SIDE = ["impressions_second_half", "is_declining_next_half"]

assert set(FINAL_FEATURES).isdisjoint(LABEL_SIDE), "LEAKAGE: overlap between features and label-side columns!"
print("Confirmed disjoint. Final feature set used for the capstone's headline numbers:")
for f in FINAL_FEATURES:
    print(" -", f)


Confirmed disjoint. Final feature set used for the capstone's headline numbers:
 - impressions_first_half
 - clicks_first_half
 - ctr_first_half
 - avg_position_first_half
 - active_days_first_half


## 4. Claim rewrite

**Boldest sentence as first drafted (too strong):** "The model predicts which pages will decline
next month."

**Rewritten in safe language:** "On March 2026 data, a Logistic Regression model trained on five
first-half activity features and evaluated on clients it never trained on achieved a measured
Precision@10 of [X]% for identifying (client, content) pairs whose impressions dropped more than
20% in the second half of the same month — an observed, decision-support result on this dataset
and time window, not a guarantee for any future month or a claim about why any individual page's
ranking changed."

The rewrite adds four things the bold version was missing: the specific dataset and window,
the fact the split held out clients, the word "measured" instead of implying a general rule, and
an explicit statement that it's decision-support, not causal or predictive of the future beyond
this test.


In [4]:
print("Before:  'The model predicts which pages will decline next month.'")
print()
print("After:   'On March 2026 data, a Logistic Regression model trained on five first-half")
print("          activity features and evaluated on held-out clients achieved a measured")
print("          Precision@10 of [X]% for is_declining_next_half -- an observed,")
print("          decision-support result on this dataset and window, not a guarantee")
print("          for any future month.'")
print()
print("Fill in [X] with your own w05_model.ipynb / cell-4-above Precision@10 result.")


Before:  'The model predicts which pages will decline next month.'

After:   'On March 2026 data, a Logistic Regression model trained on five first-half
          activity features and evaluated on held-out clients achieved a measured
          Precision@10 of [X]% for is_declining_next_half -- an observed,
          decision-support result on this dataset and window, not a guarantee
          for any future month.'

Fill in [X] with your own w05_model.ipynb / cell-4-above Precision@10 result.
